<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Simple_Machine_learning_training_example_with_CFID_descriptors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Table of contents

1. Loading CFID-3D dataset from figshare
2. Making pandas dataframe
3. Properties available, keys
4. Preparing training and target data
5. Simple regression traiing on 500 samples for formation energies
6. Visualizing training results

In [ ]:
!pip install jarvis-tools

In [ ]:
# !pip install jarvis-tools, and restart runtime in the notebook if you haven't installed it yet
from jarvis.db.figshare import data
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

cfid_3d = data('cfid_3d')

In [ ]:
len(cfid_3d)

In [ ]:
typical_data_ranges = {'formation_energy_peratom': [-5, 5], 'optb88vdw_bandgap': [0, 10], 'mbj_bandgap': [0, 10], 'bulk_modulus_kv': [0, 250], 'shear_modulus_gv': [0, 250], 'epsx': [0, 60], 'epsy': [0, 60], 'epsz': [0, 60], 'mepsx': [0, 60], 'mepsy': [0, 60], 'mepsz': [0, 60], 'n-Seebeck': [-600, 10], 'n-powerfact': [0, 5000], 'p-Seebeck': [-10, 600], 'p-powerfact': [0, 5000], 'slme': [0, 40], 'spillage': [0, 4], 'encut': [0, 2000], 'kpoint_length_unit': [0, 200], 'dfpt_piezo_max_dielectric': [0, 100], 'dfpt_piezo_max_dij': [0, 3000], 'dfpt_piezo_max_eij': [0, 10], 'ehull': [0, 1], 'electron_avg_effective_masses_300K': [0, 3], 'hole_avg_effective_masses_300K': [0, 3], 'exfoliation_energy': [0, 1000], 'magmom_oszicar': [0, 10], 'max_ir_mode': [0, 4000], 'total_energy_per_atom': [-10, 3]}

In [ ]:
typical_data_ranges

In [ ]:
df=pd.DataFrame(cfid_3d)

In [ ]:
df.columns

In [ ]:
def get_ml_data(dataframe='', ml_property='formation_energy_peratom'):
    x = []
    y = []
    jid = []
    df2 = dataframe[['desc','jid',ml_property]].replace('na',np.nan).dropna()
    for ii,i in df2.iterrows():
        if len(i['desc']) ==1557 and float(i[ml_property])!=float('inf') and i[ml_property]!='na' \
            and float(i[ml_property])<typical_data_ranges[ml_property][1] \
            and float(i[ml_property])>typical_data_ranges[ml_property][0] :
            x.append(i['desc'])
            y.append(i[ml_property])
            jid.append(i['jid'])
    return np.array(x,dtype='float'),np.array(y,dtype='float'),jid

In [ ]:
ml_property= 'formation_energy_peratom' #'bulk_modulus_kv'
x, y, jids = get_ml_data(dataframe=df, ml_property='formation_energy_peratom')

In [ ]:
# 90 %-10 % train-test split
X_train, X_test, y_train, y_test,jid_train,jid_test = train_test_split(x, y,jids, random_state=1, test_size=0.1)
print (len(X_train), len(X_test))

In [ ]:
#Let's take 500 samples only for a quick training
# For more accurate models, higher number of points and hyper-parameter optimization etc. is needed
X1=X_train[0:500]
Y1=y_train[0:500]
X2=X_test[0:500]
Y2=y_test[0:500]
print (X1.shape)

In [ ]:
#Typical hyperparameter optimized parameters in LightGBM:
#"n_estimators": 1170,
#"learning_rate": 0.15375236057119931,
#"num_leaves": 273,

In [ ]:
# GB regressor with default parameters
model = GradientBoostingRegressor()
model.fit(X1,Y1)

In [ ]:
# predict on 10% held set which was not used during training
pred=model.predict(X2)

In [ ]:
def regr_scores(pred, test):

    """
   Generic regresion scores
   Args:
       pred: predicted values
       test: held data for testing
   Returns:
        info: with metrics
   """

    rmse = np.sqrt(mean_squared_error(test, pred))
    r2 = r2_score(test, pred)
    mae = mean_absolute_error(test, pred)
    info = {}
    info["mae"] = mae
    info["rmse"] = rmse
    info["r2"] = r2
    info["test"] = test
    info["pred"] = pred
    return info

In [ ]:
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
reg_sc=regr_scores(Y2,pred)
plt.plot(reg_sc['test'],reg_sc['pred'],'.')
plt.plot(reg_sc['test'],reg_sc['test'])
plt.xlabel('DFT-Formation energy (eV/atom)')
plt.ylabel('ML-Formation energy (eV/atom)')

In [ ]:
reg_sc

Exercise: Instead of using gradientboosting, use RandomForest and Linear regression and comapre the performance